# BERT / 분류기 실험

플랫폼(`backend/app/classifier.py`)과 동일한 **mmBERT** 카테고리 분류기를 노트북에서 직접 테스트합니다.

- 14개 MMLU-Pro 카테고리 + top-k 확률
- 한국어 / 영어 샘플 질문 batch 비교
- (선택) E5 복잡도 점수 + 라우팅 시뮬레이션

**사전 준비:** `notebooks/README.md` 참고 — venv + `semantic-router-notebooks` 커널 등록

In [2]:
import time
from dataclasses import dataclass

import pandas as pd
import torch
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

# --- 플랫폼과 동일한 설정 ---
MMBERT_MODEL = "llm-semantic-router/mmbert32k-intent-classifier-merged"
E5_MODEL = "intfloat/multilingual-e5-small"
MAX_TOKENS = 512
DIFF_SCALE = 25.0
COMPLEXITY_SMALL_MAX = 0.3
COMPLEXITY_MEDIUM_MAX = 0.7

# backend/app/models_config.py 의 데모버전 매핑 (노트북에서 backend import 없이 사용)
CATEGORY_TO_COMPANY = {
    "computer science": "openai",
    "engineering": "openai",
    "philosophy": "openai",
    "law": "openai",
    "biology": "google",
    "chemistry": "google",
    "physics": "google",
    "health": "google",
    "math": "openai",
    "economics": "openai",
    "business": "openai",
    "history": "openai",
    "psychology": "openai",
    "other": "openai",
}
FALLBACK_COMPANY = "openai"

print(f"torch {torch.__version__}, device: cpu")

torch 2.5.1, device: cpu


## 1. mmBERT 카테고리 분류기 (플랫폼과 동일)

`AutoModelForSequenceClassification` — 14개 MMLU-Pro 카테고리 중 1개 선택

In [3]:
def classify_topk(
    text: str,
    model,
    tokenizer,
    top_k: int = 3,
) -> list[dict]:
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_TOKENS,
    )
    with torch.no_grad():
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1)[0]

    values, indices = probs.topk(min(top_k, probs.shape[-1]))
    results = []
    for prob, idx in zip(values, indices):
        results.append(
            {
                "label": model.config.id2label[idx.item()],
                "prob": round(prob.item(), 4),
            }
        )
    return results


def classify_label(text: str, model, tokenizer) -> str:
    return classify_topk(text, model, tokenizer, top_k=1)[0]["label"]


print("헬퍼 함수 준비 완료")

헬퍼 함수 준비 완료


In [4]:
print(f"로드 중: {MMBERT_MODEL} ...")
t0 = time.time()
mmbert_tokenizer = AutoTokenizer.from_pretrained(MMBERT_MODEL)
mmbert_model = AutoModelForSequenceClassification.from_pretrained(MMBERT_MODEL)
mmbert_model.eval()
print(f"완료 ({time.time() - t0:.1f}s)")

labels = [mmbert_model.config.id2label[i] for i in range(len(mmbert_model.config.id2label))]
print(f"카테고리 {len(labels)}개:")
for i, label in enumerate(labels):
    print(f"  {i:2d}. {label}")

로드 중: llm-semantic-router/mmbert32k-intent-classifier-merged ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

완료 (78.8s)
카테고리 14개:
   0. biology
   1. business
   2. chemistry
   3. computer science
   4. economics
   5. engineering
   6. health
   7. history
   8. law
   9. math
  10. other
  11. philosophy
  12. physics
  13. psychology


In [5]:
SAMPLE_QUESTIONS = [
    # 한국어
    "안녕하세요",
    "파이썬으로 이진 탐색 구현해줘",
    "미적분 문제 풀어줘",
    "계약서 위험 조항 검토해줘",
    "세포 분열 과정 설명해줘",
    "오늘 점심 뭐 먹을까?",
    "칸트의 정언명령을 쉽게 설명해줘",
    # 영어
    "Explain quantum entanglement in simple terms",
    "Write a Python function for binary search",
    "Summarize the causes of the French Revolution",
]

for q in SAMPLE_QUESTIONS[:3]:
    top = classify_topk(q, mmbert_model, mmbert_tokenizer, top_k=3)
    print(f"\nQ: {q}")
    for rank, item in enumerate(top, 1):
        print(f"  {rank}. {item['label']} ({item['prob']:.2%})")


Q: 안녕하세요
  1. other (99.91%)
  2. economics (0.07%)
  3. business (0.01%)

Q: 파이썬으로 이진 탐색 구현해줘
  1. computer science (99.98%)
  2. engineering (0.02%)
  3. physics (0.00%)

Q: 미적분 문제 풀어줘
  1. psychology (70.77%)
  2. economics (20.80%)
  3. biology (4.64%)


In [6]:
rows = []
for q in SAMPLE_QUESTIONS:
    top1 = classify_topk(q, mmbert_model, mmbert_tokenizer, top_k=1)[0]
    top3 = classify_topk(q, mmbert_model, mmbert_tokenizer, top_k=3)
    company = CATEGORY_TO_COMPANY.get(top1["label"], FALLBACK_COMPANY)
    rows.append(
        {
            "question": q,
            "top1_label": top1["label"],
            "top1_prob": top1["prob"],
            "company": company,
            "top3": ", ".join(f"{t['label']}({t['prob']:.0%})" for t in top3),
        }
    )

df_mmbert = pd.DataFrame(rows)
df_mmbert

,question,top1_label,top1_prob,company,top3
0,안녕하세요,other,0.9991,openai,"other(100%), economics(0%), business(0%)"
1,파이썬으로 이진 탐색 구현해줘,computer science,0.9998,openai,"computer science(100%), engineering(0%), physi..."
2,미적분 문제 풀어줘,psychology,0.7077,openai,"psychology(71%), economics(21%), biology(5%)"
3,계약서 위험 조항 검토해줘,law,0.4386,openai,"law(44%), business(25%), psychology(23%)"
4,세포 분열 과정 설명해줘,biology,1.0000,google,"biology(100%), psychology(0%), health(0%)"
5,오늘 점심 뭐 먹을까?,other,0.9999,openai,"other(100%), math(0%), economics(0%)"
6,칸트의 정언명령을 쉽게 설명해줘,philosophy,0.9932,openai,"philosophy(99%), other(0%), law(0%)"
7,Explain quantum entanglement in simple terms,engineering,0.7480,openai,"engineering(75%), physics(13%), chemistry(10%)"
8,Write a Python function for binary search,computer science,1.0000,openai,"computer science(100%), engineering(0%), philo..."
9,Summarize the causes of the French Revolution,history,0.7772,openai,"history(78%), other(21%), philosophy(1%)"


## 2. E5 복잡도 점수 (플랫폼 `complexity.py`와 동일 로직)

BERT는 아니지만, 플랫폼 auto 모드에서 **크기(s/m/l)** 를 정하는 데 함께 쓰입니다.

In [7]:
import torch.nn.functional as F

EASY_EXAMPLES = [
    "안녕하세요", "지금 몇 시야?", "1 더하기 1은?", "간단히 요약해줘",
]
HARD_EXAMPLES = [
    "이 분산 시스템에서 발생하는 데이터 정합성 문제를 CAP 정리 관점에서 분석하고 해결 방안을 제시해줘",
    "양자역학의 불확정성 원리를 일반 상대성이론과 연결지어 설명하고, 둘 사이의 이론적 긴장 관계를 논해줘",
]


def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


def embed_texts(texts: list[str], model, tokenizer) -> torch.Tensor:
    prefixed = [f"query: {t}" for t in texts]
    batch = tokenizer(
        prefixed, max_length=512, padding=True, truncation=True, return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model(**batch)
    embeddings = average_pool(outputs.last_hidden_state, batch["attention_mask"])
    return F.normalize(embeddings, p=2, dim=1)


def complexity_score(text: str, model, tokenizer, easy_ref, hard_ref) -> float:
    query_emb = embed_texts([text], model, tokenizer)
    easy_sim = (query_emb @ easy_ref.T).squeeze(0).mean().item()
    hard_sim = (query_emb @ hard_ref.T).squeeze(0).mean().item()
    diff = hard_sim - easy_sim
    return torch.sigmoid(torch.tensor(diff * DIFF_SCALE)).item()


def size_for_complexity(score: float) -> str:
    if score < COMPLEXITY_SMALL_MAX:
        return "small"
    if score < COMPLEXITY_MEDIUM_MAX:
        return "medium"
    return "large"

print("복잡도 헬퍼 준비 완료")

복잡도 헬퍼 준비 완료


In [8]:
print(f"로드 중: {E5_MODEL} ...")
t0 = time.time()
e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL)
e5_model.eval()
easy_ref = embed_texts(EASY_EXAMPLES, e5_model, e5_tokenizer)
hard_ref = embed_texts(HARD_EXAMPLES, e5_model, e5_tokenizer)
print(f"완료 ({time.time() - t0:.1f}s)")

complexity_rows = []
for q in SAMPLE_QUESTIONS:
    score = complexity_score(q, e5_model, e5_tokenizer, easy_ref, hard_ref)
    complexity_rows.append(
        {"question": q, "complexity_score": round(score, 4), "size": size_for_complexity(score)}
    )

pd.DataFrame(complexity_rows)

로드 중: intfloat/multilingual-e5-small ...


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

완료 (35.6s)


,question,complexity_score,size
0,안녕하세요,0.1268,small
1,파이썬으로 이진 탐색 구현해줘,0.6399,medium
2,미적분 문제 풀어줘,0.5158,medium
3,계약서 위험 조항 검토해줘,0.5918,medium
4,세포 분열 과정 설명해줘,0.6667,medium
5,오늘 점심 뭐 먹을까?,0.2232,small
6,칸트의 정언명령을 쉽게 설명해줘,0.4916,medium
7,Explain quantum entanglement in simple terms,0.6634,medium
8,Write a Python function for binary search,0.6399,medium
9,Summarize the causes of the French Revolution,0.6280,medium


## 3. Auto 라우팅 시뮬레이션

카테고리 → 회사 + 복잡도 → 크기를 합쳐 플랫폼 auto 모드와 같은 결정을 미리 봅니다.

In [9]:
routing_rows = []
for q in SAMPLE_QUESTIONS:
    label = classify_label(q, mmbert_model, mmbert_tokenizer)
    company = CATEGORY_TO_COMPANY.get(label, FALLBACK_COMPANY)
    score = complexity_score(q, e5_model, e5_tokenizer, easy_ref, hard_ref)
    size = size_for_complexity(score)
    routing_rows.append(
        {
            "question": q,
            "task_category": label,
            "company": company,
            "complexity_score": round(score, 4),
            "size": size,
            "route": f"{company}/{size}",
        }
    )

df_route = pd.DataFrame(routing_rows)
df_route

,question,task_category,company,complexity_score,size,route
0,안녕하세요,other,openai,0.1268,small,openai/small
1,파이썬으로 이진 탐색 구현해줘,computer science,openai,0.6399,medium,openai/medium
2,미적분 문제 풀어줘,psychology,openai,0.5158,medium,openai/medium
3,계약서 위험 조항 검토해줘,law,openai,0.5918,medium,openai/medium
4,세포 분열 과정 설명해줘,biology,google,0.6667,medium,google/medium
5,오늘 점심 뭐 먹을까?,other,openai,0.2232,small,openai/small
6,칸트의 정언명령을 쉽게 설명해줘,philosophy,openai,0.4916,medium,openai/medium
7,Explain quantum entanglement in simple terms,engineering,openai,0.6634,medium,openai/medium
8,Write a Python function for binary search,computer science,openai,0.6399,medium,openai/medium
9,Summarize the causes of the French Revolution,history,openai,0.6280,medium,openai/medium


## 4. 다른 BERT 모델 추가 실험

아래 `EXTRA_MODELS`에 HuggingFace **`ForSequenceClassification`** 파인튜닝 모델 ID를 추가하면
같은 질문 리스트에 대해 batch 비교할 수 있습니다.

> `bert-base-uncased` 같은 **베이스 모델**은 분류 헤드가 없어 이 섹션과 호환되지 않습니다.

In [10]:
# 비교할 추가 모델 ID (필요 시 직접 추가)
EXTRA_MODELS: list[str] = [
    # "example/intent-classifier",  # AutoModelForSequenceClassification 지원 모델만
]

ALL_MODELS = [MMBERT_MODEL, *EXTRA_MODELS]


@dataclass
class LoadedClassifier:
    name: str
    model: object
    tokenizer: object


def load_classifiers(model_ids: list[str]) -> list[LoadedClassifier]:
    loaded: list[LoadedClassifier] = []
    for model_id in model_ids:
        print(f"로드: {model_id}")
        tok = AutoTokenizer.from_pretrained(model_id)
        mdl = AutoModelForSequenceClassification.from_pretrained(model_id)
        mdl.eval()
        loaded.append(LoadedClassifier(model_id, mdl, tok))
    return loaded


def compare_models(questions: list[str], classifiers: list[LoadedClassifier]) -> pd.DataFrame:
    rows = []
    for q in questions:
        row: dict = {"question": q}
        for clf in classifiers:
            short = clf.name.split("/")[-1][:30]
            label = classify_label(q, clf.model, clf.tokenizer)
            prob = classify_topk(q, clf.model, clf.tokenizer, top_k=1)[0]["prob"]
            row[f"{short}_label"] = label
            row[f"{short}_prob"] = prob
        rows.append(row)
    return pd.DataFrame(rows)


if EXTRA_MODELS:
    extra_classifiers = load_classifiers(EXTRA_MODELS)
    compare_models(SAMPLE_QUESTIONS, [LoadedClassifier(MMBERT_MODEL, mmbert_model, mmbert_tokenizer), *extra_classifiers])
else:
    print("EXTRA_MODELS가 비어 있습니다. 비교할 모델 ID를 추가한 뒤 이 셀을 다시 실행하세요.")

EXTRA_MODELS가 비어 있습니다. 비교할 모델 ID를 추가한 뒤 이 셀을 다시 실행하세요.


In [11]:
# (선택) 결과를 CSV로 저장
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

df_route.to_csv(out_dir / "routing_simulation.csv", index=False)
print(f"저장: {out_dir / 'routing_simulation.csv'}")

저장: outputs/routing_simulation.csv
